In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from docent.data_models.agent_run import AgentRun
from docent._loader.load_inspect import load_inspect_eval
from docent.sdk.client import DocentClient
os.environ['ENV_TYPE'] = 'swe-bench'
# from docent._loader.load_inspect import load_swebench

client = DocentClient(server_url="http://localhost:8889", web_url="http://localhost:4001", email="sherwinxyu@gmail.com")
fgs = client.list_framegrids()
print(fgs)


if fgs:
    fg_id = fgs[1]['id']
    print('fg', fg_id, fgs[1]['name'])
else:
    fg_id = client.create_framegrid(name="swe-bench example")

SWE_BENCH_LOGS: dict[str, str | tuple[str, dict[str, str]]] = {
    # "swebench-sonnet37-old": f"/home/ubuntu/artifacts/vincent/swe_bench_logs/2025-04-09T21-09-59+00-00_swe-bench_8AcW4AHxbhgtoqEbe5FQcT.eval",
    # "swebench-sonnet37-new": f"/home/ubuntu/artifacts/vincent/swe_bench_logs/2025-04-10T21-39-15+00-00_swe-bench_TZrCQjagGBxzuSrXnE3fqj.eval",
    "swebench-sonnet35-tools": "~/projects/docent-artifacts/eval1.eval",
    "swebench-sonnet37-tools": "~/projects/docent-artifacts/eval2.eval",
}
def load_swebench() -> list[AgentRun]:
    return load_inspect_eval(SWE_BENCH_LOGS)
# agent_runs = load_swebench()
# client.add_agent_runs(fg_id, agent_runs)

TRANSCRIPTS = load_swebench()

In [ ]:
fg_id

In [ ]:
from docent.data_models.agent_run import AgentRun

def find_by_expid(transcripts: list[AgentRun], expid: str) -> AgentRun:
    for t in transcripts:
        if t.metadata.get("experiment_id") == expid:
            return t
    raise ValueError(f"No transcript found for experiment ID {expid}")

def printw(s: str):
    LINE_LENGTH = 150
    lines = s.split('\n')
    for line in lines:
        for i in range(0, len(line), LINE_LENGTH):
            print(line[i:i+LINE_LENGTH])

In [ ]:
from docent.data_models.transcript import (
    MULTI_BLOCK_CITE_INSTRUCTION,
)
from docent._llm_util.prod_llms import get_llm_completions_async
from docent._llm_util.providers.preferences import PROVIDER_PREFERENCES

# Multi-Stage

### Run it

In [ ]:
import os
from docent.data_models.agent_run import AgentRun


os.environ['ENV_TYPE'] = 'swe-bench'



In [ ]:

all_sample_ids: set[str] = {
    "astropy__astropy-14096", "django__django-11551", "django__django-11239", "django__django-14351",
    "django__django-11299", "django__django-11749", "django__django-12193", "django__django-14404",
}
print(TRANSCRIPTS[0:4])
TRANSCRIPT_PAIRS: dict[str, list[AgentRun]] = {}
for sample_id in all_sample_ids:
    sample_transcripts = [t for t in TRANSCRIPTS if t.metadata.get("sample_id") == sample_id]
    if len(sample_transcripts) != 2:
        continue
    first, second = find_by_expid(sample_transcripts, 'swebench-sonnet37-tools'), find_by_expid(sample_transcripts, 'swebench-sonnet35-tools')
    TRANSCRIPT_PAIRS[sample_id] = [first, second]

len(TRANSCRIPT_PAIRS)

len(TRANSCRIPT_PAIRS)
msgs = next(iter(TRANSCRIPT_PAIRS.items()))[1][0].transcripts['default'].messages

for i in range(7):
    print("~~{}~ ~{}~~\n{}\n\n".format(i, msgs[i].role, msgs[i].text))


In [ ]:
# Benchmark - time for running ONE pair of diffs is ~ 1m 15s
# Run 2 - 2m 23s
pair = next(iter(TRANSCRIPT_PAIRS.items()))[1]
from docent._ai_tools.diffs.llm_diff_summaries import compute_transcript_diff
x =  await compute_transcript_diff(pair[0], pair[1])
print(x.claims[0].model_dump_json(indent=2))


# swe_diffs = await extract_states_and_diffs()
# swe_diffs.keys()


In [ ]:
from docent._ai_tools.diffs.llm_message_summaries import get_llm_output_for_transcript_to_message_summaries
text = await get_llm_output_for_transcript_to_message_summaries(pair[0])

In [ ]:
from docent._ai_tools.diffs.llm_message_summaries import llm_output_to_message_summaries
messages = llm_output_to_message_summaries(text)

### My approach - Claim, shared_context, actions

In [ ]:
from docent._ai_tools.diffs.llm_diff_summaries import get_llm_output_compare_transcript_states_v2

messages_0 = llm_output_to_message_summaries(await get_llm_output_for_transcript_to_message_summaries(pair[0]))
messages_1 = llm_output_to_message_summaries(await get_llm_output_for_transcript_to_message_summaries(pair[1]))

diff_text = await get_llm_output_compare_transcript_states_v2(pair[0], pair[1], messages_0, messages_1)
print(diff_text)

In [ ]:
for k, pair in TRANSCRIPT_PAIRS.items():
    print(k, '---', pair[0].metadata.get("experiment_id"), pair[1].metadata.get("experiment_id"))
pairs = [pair for _k, pair in TRANSCRIPT_PAIRS.items()][:3]
diffs = [await compute_transcript_diff(pair[0], pair[1]) for pair in pairs]
claims = [c for diff in diffs for c in diff.claims]

In [ ]:
from docent._ai_tools.diffs.llm_diff_summaries import llm_summarize_transcript_title
pair = list(TRANSCRIPT_PAIRS.values())[0]
title = await llm_summarize_transcript_title(pair[0], pair[1])
print(title)
claims[0].model_dump()

### Generate the json for FE consumption

In [ ]:
from docent._ai_tools.diffs.llm_diff_summaries import get_llm_output_compare_transcript_states_v2, _parse_llm_output_to_claims
diff_text = await get_llm_output_compare_transcript_states_v2(pair[0], pair[1], messages_0, messages_1)
parsed_claims = _parse_llm_output_to_claims(diff_text)
[pc.model_dump() for pc in parsed_claims]

# diff = await compute_transcript_diff(pair[0], pair[1])
# [print(c.claim_summary)for c in diff.claims] 

# Clustering Diffs

In [ ]:
from docent._ai_tools.clustering.cluster_diffs import cluster_diff_claims
from docent._ai_tools.diffs.models import SQLATranscriptDiff, SQLADiffsReport


# LLM-based Claim Interestingness Ranking

This section tests the LLM-based ranking of claims by interestingness.

In [ ]:
from docent._ai_tools.diffs.llm_rank_diffs import rank_claims_by_interestingness
from docent._ai_tools.diffs.models import Claim

# Example claims for testing
# claims = [
#     Claim(
#         claim_summary="Agent 1 uses a general dtype fix, Agent 2 tries targeted fixes.",
#         shared_context="Both agents are fixing a numpy dtype bug.",
#         agent_1_action="Implements dtype kind-based solution.",
#         agent_2_action="Implements multiple targeted fixes.",
#         evidence="Agent 1: [T0B10], Agent 2: [T1B12][T1B13]"
#     ),
#     Claim(
#         claim_summary="Agent 1 adds formal unit tests, Agent 2 only creates a manual test script.",
#         shared_context="Both agents are verifying their fixes.",
#         agent_1_action="Writes pytest unit tests.",
#         agent_2_action="Runs manual script.",
#         evidence="Agent 1: [T0B20], Agent 2: [T1B22]"
#     ),
#     # Add more claims as needed
# ]


In [ ]:
import asyncio
rankings = asyncio.run(rank_claims_by_interestingness(claims))
for i, (score, justification) in enumerate(rankings, 1):
    print(f"Claim {i}: Score={score}\nJustification: {justification}\n")

# Attach scores to claims and sort
scored_claims = [
    (score, justification, claim, idx)
    for (score, justification), claim, idx in zip(rankings, claims, range(len(claims)))
]
scored_claims.sort(reverse=True, key=lambda x: x[0])
print('--------------------------------')
for score, justification, claim, idx in scored_claims:
    print('\n\n')
    print(f"#{idx} -- Score: {score}")
    print(f"Summary: {claim.claim_summary}")
    print(f"Justification: {justification}")
    print(f"ctx: {claim.shared_context}")
    print(f"A1: {claim.agent_1_action}")
    print(f"A2: {claim.agent_2_action}")
    print(f"nEvidence: {claim.evidence}")